## Data Ingestion

In [ ]:

# ## GeeksforGeeks Loader – Dynamic Content Extraction (Guaranteed >0 Docs)

from langchain_community.document_loaders import WebBaseLoader
import bs4
import re

urls = [
    "https://www.geeksforgeeks.org/machine-learning/machine-learning/",
    "https://www.geeksforgeeks.org/machine-learning/supervised-machine-learning/",
    "https://www.geeksforgeeks.org/machine-learning/regularization-in-machine-learning/",
    "https://www.geeksforgeeks.org/machine-learning/confusion-matrix-machine-learning/",
    "https://www.geeksforgeeks.org/machine-learning/ml-bias-variance-trade-off/",
    "https://www.geeksforgeeks.org/machine-learning/underfitting-and-overfitting-in-machine-learning/",
    "https://www.geeksforgeeks.org/data-science/what-is-gradient-descent/",
    "https://www.geeksforgeeks.org/machine-learning/ml-common-loss-functions/",
    "https://www.geeksforgeeks.org/machine-learning/ml-classification-vs-regression/",
    "https://www.geeksforgeeks.org/machine-learning/decision-tree-introduction-example/",
    "https://www.geeksforgeeks.org/machine-learning/a-comprehensive-guide-to-ensemble-learning/",
    "https://www.geeksforgeeks.org/machine-learning/what-is-feature-engineering/",
    "https://www.geeksforgeeks.org/machine-learning/hyperparameter-tuning/",
]

def extract_main_article(html):
    """Dynamic extractor: Find div with most <p> tags (the article body)"""
    soup = bs4.BeautifulSoup(html, "html.parser")
    
    # Find all divs and score by number of <p> children
    candidates = soup.find_all("div")
    if not candidates:
        return soup.get_text()  # Fallback to full text
    
    best_div = max(candidates, key=lambda d: len(d.find_all("p")), default=soup)
    
    # Clean the best div
    for tag in best_div(["script", "style", "nav", "header", "footer", "aside", "iframe", "img"]):
        tag.decompose()
    
    # Extract text from key elements
    text_parts = []
    for elem in best_div.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "blockquote"]):
        text = elem.get_text(separator=" ", strip=True)
        if len(text) > 15:  # Meaningful length
            text_parts.append(text)
    
    full_text = "\n\n".join(text_parts)
    full_text = re.sub(r'\n{3,}', '\n\n', full_text.strip())
    full_text = re.sub(r'\s+', ' ', full_text)  # Normalize whitespace
    
    return full_text

# Load with custom extractor (no bs_kwargs needed)
loader = WebBaseLoader(
    web_paths=urls,
    # Custom post-load processing via a simple override
)

docs = loader.load()

# Apply extractor to each
clean_docs = []
for d in docs:
    extracted = extract_main_article(d.page_content)
    if len(extracted) > 500:  # Only keep substantial content
        d.page_content = extracted
        # Try to grab title
        soup = bs4.BeautifulSoup(d.page_content, "html.parser")  # Reuse for title
        title_elem = soup.find("h1") or soup.find("title")
        d.metadata["title"] = title_elem.get_text(strip=True) if title_elem else "ML Article"
        clean_docs.append(d)
    else:
        print(f"Skipped {d.metadata.get('source')}: too short ({len(extracted)} chars)")

print(f"Successfully loaded & cleaned {len(clean_docs)} documents")


Successfully loaded & cleaned 13 documents


## Chunking Data

In [196]:
# Chunking the Data
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
documents = text_splitter.split_documents(clean_docs)

In [42]:
print(documents)

[Document(metadata={'source': 'https://www.geeksforgeeks.org/machine-learning/machine-learning/', 'title': 'ML Article', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en-US'}, page_content='Machine Learning Tutorial - GeeksforGeeks\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content'), Document(metadata={'source': 'https://www.geeksforgeeks.org/machine-learning/machine-learning/', 'title': 'ML Article', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en-US'}, page_content='Tutor

### Using Gemini token from .env

In [177]:
import os
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("Gemini API key not found in .env under 'GEMINI_API_KEY'")

### Intializing Embedding model

In [197]:
# Creating Embeddings and Vector Store
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
# Initialize the model
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2") # 384 dimensions


### Loading VectorDB

In [198]:
# Create the vector store
vectordb = Chroma.from_documents(
    documents=documents,
    collection_name="rag-chroma",
    embedding=embedding_model,
)
retriever = vectordb.as_retriever(search_kwargs={"k": 6})

### Loading Multiple models from Gemini

In [50]:
# List available models (for debugging)
import google.generativeai as genai
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-lat

In [199]:
# === Simple Gemini Wrapper for LangChain ===
import os
from dotenv import load_dotenv

from langchain_core.runnables import RunnableLambda

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Gemini API key not found in .env under 'GEMINI_API_KEY'")

genai.configure(api_key=api_key)

_GEMINI_MODEL_ID = "gemma-3-1b-it"
_generation_config = genai.types.GenerationConfig(
    temperature=0.3,
    top_p=0.9,
    max_output_tokens=512,
)
_model = genai.GenerativeModel(_GEMINI_MODEL_ID)

def _gemini_call(inputs, **kwargs) -> str:
    if isinstance(inputs, dict):
        prompt_text = "\n\n".join(f"{k}: {v}" for k, v in inputs.items())
    else:
        prompt_text = str(inputs)
    try:
        resp = _model.generate_content(
            prompt_text,
            generation_config=_generation_config,
            safety_settings={  # Optional: relax safety if needed
                genai.types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: genai.types.HarmBlockThreshold.BLOCK_NONE,
            }
        )

        # === DEBUG: Print token usage ===
        if hasattr(resp, 'usage_metadata'):
            print(f"[DEBUG] Prompt tokens: {resp.usage_metadata.prompt_token_count}")
            print(f"[DEBUG] Output tokens: {resp.usage_metadata.candidates_token_count}")
            print(f"[DEBUG] Total tokens: {resp.usage_metadata.total_token_count}")

        # === CASE 1: Normal response with text ===
        if resp.candidates and resp.candidates[0].content.parts:
            text = resp.candidates[0].content.parts[0].text.strip()
            return text if text else "Empty response from model."

        # === CASE 2: No text generated (e.g., MAX_TOKENS) ===
        candidate = resp.candidates[0]
        finish_reason = candidate.finish_reason

        if finish_reason == 2:  # MAX_TOKENS
            return "[TRUNCATED] Response cut off due to max_output_tokens. Try reducing context size."
        elif finish_reason == 3:  # SAFETY
            return "[BLOCKED] Response blocked for safety."
        else:
            return f"[NO OUTPUT] Finish reason: {finish_reason}"

    except Exception as e:
        return f"[ERROR] {str(e)}"

# Rebuild LLM
llm = RunnableLambda(_gemini_call)

#### From each doc trying to extract metadata like if present page.no. to add to end of chunk

In [187]:
# ==== RAG Prompt & Chain ====
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(clean_docs):
    # Helpful for page-aware citations if metadata exists
    def one(d):
        pg = d.metadata.get("page", None)
        tag = f" [page {pg}]" if pg is not None else ""
        return d.page_content.strip() + tag
    return "\n\n---\n\n".join(one(d) for d in clean_docs)
    

### Zero-shot Prompt

In [188]:

# Define a plain (non-chat) prompt template to avoid role tags in output
prompt = PromptTemplate.from_template(
    """You are an expert in machine learning. Use ONLY the provided context to answer the question.
If the answer is not in the context, respond exactly: I don't know.

Answer in a clear, concise paragraph. Do not use bullet points. Do not add external knowledge.

Question: {question}

Context: {context}

Answer:"""
)

# Keep your retriever the same; build a chain that accepts explicit context
generation_chain = prompt | llm | StrOutputParser()



## Few-shot prompt

In [189]:

# --- Few-shot: two format exemplars, still "use only context" ---
prompt = PromptTemplate.from_template(
    """You are given a question and some context. Use ONLY the context.
If the answer is not in the context, respond exactly: I don't know.

Answer in a concise, natural paragraph (not bullets).

### EXAMPLE
Question: What is overfitting?

Context: Overfitting occurs when a model learns the training data too well, including noise. This leads to poor performance on new data.

Answer: Overfitting occurs when a model learns the training data too well, including noise, leading to poor performance on new data.

### TASK
Question: {question}

Context: {context}

Answer:"""
)
generation_chain_fewshot = prompt | llm | StrOutputParser()


## Chain-of-thought (CoT) prompt

In [200]:

# --- CoT: brief step-by-step, but return a clean <final>...</final> block we can parse ---
prompt = PromptTemplate.from_template(
    """You are an expert in machine learning. Use ONLY the context to answer the question.
If the answer is not in the context, respond exactly: I don't know.

Think step-by-step VERY briefly (1-2 sentences max), then give the final answer inside <final>...</final>.
The final answer must be a concise paragraph — no bullets, no extra text.

Question: {question}

Context: {context}

Reasoning (brief):
1) Identify key facts from context.
2) Formulate a direct, natural answer.

<final>
<!-- Final answer only -->
</final>
"""
)
generation_chain_cot = prompt | llm | StrOutputParser()




#### This is mainly extracting content between final tags during  COT since it provides reasoning and with final tag.

In [162]:
import re
def extract_final(text: str) -> str:
    m = re.search(r"<final>(.*?)</final>", text, flags=re.DOTALL|re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return text.strip()  # fallback if tags missing

def generate_answer(chain, q: str) -> str:
    # Retrieve & format context exactly like your base flow
    docs = retriever.invoke(q)
    ctx = format_docs(docs)
    out = chain.invoke({"question": q, "context": ctx})
    return out, ctx


In [191]:
# Helper to run once, show the exact retrieved context, and generate
def answer_with_trace(q: str):
    docs = retriever.invoke(q)         # same retriever as before
    ctx = format_docs(docs)                            # exactly what goes into the prompt
    ans = generation_chain.invoke({"question": q, "context": ctx})

    print("\nuser question:\n")
    print(q)
    print("\nretrieved context:\n")
    print(ctx if len(ctx) < 4000 else ctx[:4000] + "\n...[truncated]...")
    print("\nllm output:\n")
    print(ans)

# ==== Ask questions ====
question = "what is machine learning?"
answer_with_trace(question)

[DEBUG] Prompt tokens: 1015
[DEBUG] Output tokens: 71
[DEBUG] Total tokens: 1086

user question:

what is machine learning?

retrieved context:

Machine learning is a branch of Artificial Intelligence that focuses on developing models and algorithms that let computers learn from data without being explicitly programmed for every task. In simple words, ML teaches the systems to think and understand like humans by learning from the data.Try our ongoing free course Data Science Skillup with weekly topic coverage, notes, daily quizzes and coding problems.Machine Learning is mainly divided into three core types: Supervised, Unsupervised and Reinforcement Learning along with two additional types, Semi-Supervised and Self-Supervised Learning.Supervised Learning: Trains models on labeled data to predict or classify new, unseen data.Unsupervised Learning: Finds patterns or groups in

---

Machine learning is a branch of Artificial Intelligence that focuses on developing models and algorithms th

### Loading Ground Truth Dataset from Huggingface

In [128]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
df = pd.read_csv("hf://datasets/prsdm/Machine-Learning-QA-dataset/ML-101-QandA.csv")

In [163]:

# === Load evaluation dataset ===

# Reference question/answer columns
questions = df["Question"].tolist()
gold_answers = df["Answer"].tolist()

# Embedding model for semantic comparison
eval_embedder = SentenceTransformer("all-MiniLM-L6-v2")

### Evaluation metrics

In [201]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# === Load evaluation dataset (only first 10 rows) ===
sample_df = df.head(10)  #
questions = sample_df["Question"].tolist()
gold_answers = sample_df["Answer"].tolist()

# Embedding model for semantic comparison
eval_embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Storage for metric results
results = []

for q, gold in tqdm(zip(questions, gold_answers), total=len(questions), desc="Evaluating RAG (1 samples)"):
    # --- Retrieve context ---
    retrieved_docs = retriever.invoke(q)
    retrieved_texts = [d.page_content for d in retrieved_docs]
    context = "\n\n".join(retrieved_texts)
    
    # --- Generate answer ---
    gen_answer = generation_chain_cot.invoke({"question": q, "context": context})

    # --- Compute embeddings ---
    q_emb = eval_embedder.encode([q])
    gold_emb = eval_embedder.encode([gold])
    gen_emb = eval_embedder.encode([gen_answer])
    ctx_embs = eval_embedder.encode(retrieved_texts)

    # --- Retrieval metrics ---
    ctx_rel = float(np.mean(cosine_similarity(q_emb, ctx_embs)))   # Context Relevance
    ctx_rec = float(np.max(cosine_similarity(gold_emb, ctx_embs))) # Context Recall

    # --- Generation metrics ---
    faith = float(np.mean(cosine_similarity(gen_emb, ctx_embs)))   # Faithfulness (Groundedness)
    ans_rel = float(cosine_similarity(gen_emb, gold_emb)[0][0])    # Answer Relevance

    # --- End-to-end metric ---
    ans_corr = (2 * faith * ans_rel) / (faith + ans_rel + 1e-9)    # Answer Correctness

    results.append({
        "Question": q,
        "Context Relevance": round(ctx_rel, 3),
        "Context Recall": round(ctx_rec, 3),
        "Faithfulness": round(faith, 3),
        "Answer Relevance": round(ans_rel, 3),
        "Answer Correctness": round(ans_corr, 3)
    })

# === Average metrics ===
import pandas as pd

results_df = pd.DataFrame(results)
avg_metrics = results_df.drop(columns=["Question"]).mean().round(3)

print("\n=== Average RAG Evaluation Metrics (10 samples) ===\n")
print("\n Metrics")
print(avg_metrics.to_frame().T)


Evaluating RAG (1 samples):   0%|          | 0/10 [00:00<?, ?it/s]

[DEBUG] Prompt tokens: 1151
[DEBUG] Output tokens: 46
[DEBUG] Total tokens: 1197


Evaluating RAG (1 samples):  10%|█         | 1/10 [00:01<00:12,  1.38s/it]

[DEBUG] Prompt tokens: 1145
[DEBUG] Output tokens: 90
[DEBUG] Total tokens: 1235


Evaluating RAG (1 samples):  20%|██        | 2/10 [00:02<00:11,  1.41s/it]

...

[DEBUG] Prompt tokens: 1066
[DEBUG] Output tokens: 63
[DEBUG] Total tokens: 1129


Evaluating RAG (1 samples):  90%|█████████ | 9/10 [00:10<00:01,  1.15s/it]

[DEBUG] Prompt tokens: 1268
[DEBUG] Output tokens: 64
[DEBUG] Total tokens: 1332


Evaluating RAG (1 samples): 100%|██████████| 10/10 [00:12<00:00,  1.20s/it]


=== Average RAG Evaluation Metrics (10 samples) ===


 Metrics
   Context Relevance  Context Recall  Faithfulness  Answer Relevance  \
0              0.722           0.777         0.752             0.808   

   Answer Correctness  
0               0.771  
